# ADoRE-DoE Phase 1：Screening 範例

因子 → 設計 → 匯出 → 填入 response → 主效應 → 篩選 → 圖表 → 匯出候補因子。response 由**帶 seed 的合成資料**產生（僅示範）。安裝：`pip install -e .`

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import adore_doe as ad
os.makedirs('output', exist_ok=True)
SEED = 2026

## 1. 因子與 PB 設計（8 因子、12 runs、3 center points、隨機化）

In [ ]:
factors = [ad.Factor(n, lo, hi) for n, lo, hi in
           [("A",0.1,0.3),("B",10,20),("C",1,5),("D",50,90),("E",0,1),("F",100,200),("G",5,15),("H",2,8)]]
design = ad.screening.plackett_burman(factors, runs=12, center_points=3, randomize=True, seed=SEED)
print(design.summary())
design.to_csv('output/screening_design.csv')
design.to_dataframe(order='run').head(6)

PB 只用於主效應篩選；N=12 時二因子交互作用**部分**混淆於多個主效應（±1/3）。

## 2. 填入 response（實驗者依 RunOrder 記錄，用 `order='run'`）

In [ ]:
rng = np.random.default_rng(SEED); X = design.coded
truth = 20 + 6*X[:,0] - 4*X[:,2] + 1.5*X[:,5]
y_std = truth + 0.5*rng.standard_normal(len(truth))
design.add_response('Yield', y_std[np.argsort(design.run_order)], order='run')
design.add_response('Purity', 80 + 0.1*y_std[np.argsort(design.run_order)], order='run')   # 多響應共用同一 X
design.to_csv('output/screening_with_results.csv')
design = ad.ExperimentDataset.from_csv('output/screening_with_results.csv')
design.response_names

## 3. 主效應與篩選

In [ ]:
effects = ad.screening.main_effects(design, 'Yield', alpha=0.05)
print(f"error df={effects.df_error}  t-limit={effects.t_limit:.4f}  Bonferroni={effects.bonferroni_limit:.4f}")
effects.table.round(4)

In [ ]:
res_t = ad.screening.screen(effects)                         # t-limit 規則
res_b = ad.screening.screen(effects, criterion='bonferroni') # 控制 family-wise error
print('t-limit    :', res_t.selected_factors)
print('Bonferroni :', res_b.selected_factors)
res_b.history()

## 4. 圖表

In [ ]:
res_b.plot_pareto(save_path='output/pareto.png'); plt.show()

In [ ]:
res_b.plot_half_normal(save_path='output/half_normal.png'); plt.show()

In [ ]:
res_b.plot_abs_effects(save_path='output/abs_effects.png'); plt.show()

## 5. 匯出候補因子（給 RSM Design；不是定論）

In [ ]:
res_b.export_selected('output/selected_factors.json'); print(res_b.selected_factors)

## 6. 教科書對照：Montgomery 2⁴ 過濾速率範例（Fractional Factorial 模組）

In [ ]:
y = [45,71,48,65,68,60,80,65,43,100,45,104,75,86,70,96]       # 標準順序
ff = ad.screening.fractional_factorial([ad.Factor(c,-1,1) for c in 'ABCD'])
ff.add_response('rate', y)
e = ad.screening.main_effects(ff, 'rate')
print(dict(zip(e.table.Factor, e.table.Effect)))
print(dict(zip(e.dummy_table.Factor, e.dummy_table.Effect)))   # 交互作用對比
e.plot_half_normal(label='all'); plt.show()

## 7. 飽和 PB（無誤差 df）：退回 Lenth，不產生 p 值

In [ ]:
import warnings
sat = ad.screening.plackett_burman([ad.Factor(c,0,1) for c in 'ABCDEFG'], runs=8, randomize=True, seed=1)
Xs = sat.coded
sat.add_response('Y', 5 + 4*Xs[:,0] - 3*Xs[:,3] + 0.2*np.random.default_rng(1).standard_normal(8))
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    es = ad.screening.main_effects(sat, 'Y'); rs = ad.screening.screen(es)
print(rs.method, rs.selected_factors, [str(x.message)[:70] for x in w][:2])
rs.plot_pareto(); plt.show()